# VibeShift Inference: Transform Mel to Rock & Convert to Audio

This notebook shows how to:
1. Load a trained checkpoint
2. Load an input mel spectrogram
3. Transform it to rock style using flow matching
4. Convert the output mel back to audio

## Step 1: Import Dependencies

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from models.dit import DiT
from models.flow import FlowMatching
from training.training import TrainingConfig

# Setup paths
root = Path(r"c:\Users\Dhanuja\Desktop\Vibeshift\VibeShift")
checkpoint_dir = Path(r"C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\notebooks\checkpoints\paired_training")
data_dir = root / "data" / "output"
output_dir = root / "app" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"✓ Dependencies loaded")
print(f"  Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"  Checkpoint dir: {checkpoint_dir}")

## Step 2: Load Trained Checkpoint

In [ ]:
# Find the best checkpoint
checkpoint_path = Path(r"C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\notebooks\checkpoints\paired_training\best_model.pt")

if not checkpoint_path.exists():
    # Fall back to the latest checkpoint
    checkpoints = sorted(checkpoint_dir.glob("checkpoint_epoch_*.pt"))
    if checkpoints:
        checkpoint_path = checkpoints[-1]
    else:
        raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")

print(f"Loading checkpoint: {checkpoint_path}")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location='cpu')
config_dict = checkpoint['config']

print(f"\n✓ Checkpoint loaded:")
print(f"  Epoch: {checkpoint['epoch']}")
print(f"  Loss: {checkpoint['avg_loss']:.4f}")

## Step 3: Initialize Model with Checkpoint Weights

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Recreate model with saved config
dit = DiT(
    in_channels=config_dict.get('in_channels', 1),
    patch_height=config_dict.get('patch_height', 10),
    patch_width=config_dict.get('patch_width', 4),
    embed_dim=config_dict.get('embed_dim', 256),
    num_blocks=config_dict.get('num_blocks', 4),
    num_heads=config_dict.get('num_heads', 4),
    hidden_dim=config_dict.get('hidden_dim', 1024),
    num_genres=config_dict.get('num_genres', 2),
    dropout=config_dict.get('dropout', 0.1),
).to(device)

flow = FlowMatching(dit).to(device)

# Load trained weights
flow.load_state_dict(checkpoint['model_state'])
flow.eval()  # Set to evaluation mode

print(f"\n✓ Model initialized and weights loaded")
print(f"  Device: {device}")
print(f"  Parameters: {sum(p.numel() for p in flow.parameters()):,}")

## Step 4: Load Input Mel Spectrogram

Three options:
1. Load pre-computed mel from disk
2. Load FLAC audio through MIDI conversion pipeline (Audio → MIDI → Audio → Mel)
3. Convert an audio file to mel on-the-fly


In [ ]:
# Option 0: Load FLAC and process through MIDI pipeline
flac_path = Path(r"C:\Users\Dhanuja\Desktop\Vibeshift\rock_instru\00001_instrum.flac")

if flac_path.exists():
    print(f"Processing FLAC through MIDI pipeline: {flac_path.name}")
    print("-" * 70)
    
    # Import audio-to-MIDI converter
    import sys
    sys.path.insert(0, str(root))
    from utills.audio_midi_converter import AudioMIDIConverter
    import tempfile
    
    # Step 1: Load FLAC audio
    print("Step 1: Loading FLAC audio...")
    waveform, sr = torchaudio.load(str(flac_path))
    print(f"  ✓ Loaded: {waveform.shape[0]} channel(s), {sr} Hz, {waveform.shape[1] / sr:.2f}s")
    
    # Convert stereo to mono
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
        print(f"  ✓ Converted to mono")
    
    # Resample to 22050 Hz for MIDI converter
    if sr != 22050:
        resampler = T.Resample(sr, 22050)
        waveform = resampler(waveform)
        sr = 22050
        print(f"  ✓ Resampled to {sr} Hz")
    
    # Save temporarily as WAV for MIDI converter
    temp_wav = Path(tempfile.gettempdir()) / "temp_flac_input.wav"
    torchaudio.save(str(temp_wav), waveform, sr)
    
    # Step 2: Convert audio to MIDI
    print("\nStep 2: Converting audio to MIDI (using pitch detection)...")
    # Use pysynth engine instead of fluidsynth (no external dependencies)
    converter = AudioMIDIConverter(sr=sr, pitch_method="basic_pitch", synth_engine="pysynth")
    
    # Convert to MIDI file
    temp_midi = Path(tempfile.gettempdir()) / "temp_flac_output.mid"
    midi_metadata = converter.audio_to_midi(
        audio_path=temp_wav,
        output_path=temp_midi,
        bpm=120,
        confidence_threshold=0.1
    )
    print(f"  ✓ Detected MIDI: {midi_metadata['total_notes']} notes")
    
    # Step 3: Convert MIDI back to audio
    print("\nStep 3: Converting MIDI back to audio (synthesis)...")
    temp_audio_output = Path(tempfile.gettempdir()) / "temp_flac_synthesis.wav"
    midi_metadata_output = converter.midi_to_audio(
        midi_path=str(temp_midi),
        output_path=str(temp_audio_output),
        duration=None
    )
    
    # Load the synthesized audio
    audio_from_midi, _ = torchaudio.load(str(temp_audio_output))
    audio_from_midi = audio_from_midi.squeeze().numpy()
    print(f"  ✓ Synthesized audio: {len(audio_from_midi)} samples, {len(audio_from_midi) / sr:.2f}s")
    
    # Step 4: Convert audio to mel spectrogram
    print("\nStep 4: Converting audio to mel spectrogram...")
    mel_transform = T.MelSpectrogram(
        sample_rate=sr,
        n_fft=1024,
        hop_length=256,
        n_mels=100,
        f_min=0,
        f_max=8000
    )
    
    audio_tensor = torch.from_numpy(audio_from_midi).float().unsqueeze(0)
    input_mel = mel_transform(audio_tensor)
    sample_rate = sr
    
    print(f"  ✓ Generated mel spectrogram: {input_mel.shape}")
    print(f"\n✅ FLAC processing complete!")
    print(f"  Final mel shape: {input_mel.shape}")
    
    # Cleanup temp files
    temp_wav.unlink(missing_ok=True)
    temp_midi.unlink(missing_ok=True)
    
elif flac_path.exists() == False:
    # Option 1: Load pre-computed mel from data directory
    mel_file = Path(r"C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\rock_mel\00000_instrum_mel.pt")
    
    if mel_file.exists():
        data = torch.load(mel_file)
        if isinstance(data, dict):
            input_mel = data['mel']
            sample_rate = data.get('sample_rate', 24000)
        else:
            input_mel = data
            sample_rate = 24000
        
        print(f"✓ Loaded mel from {mel_file.name}")
    else:
        # Option 2: Convert audio to mel
        audio_path = root / "data" / "nonrock_audio_files" / "example.wav"  # Change this
        
        if not audio_path.exists():
            raise FileNotFoundError(f"Audio file not found: {audio_path}")
        
        waveform, sr = torchaudio.load(audio_path)
        
        # Convert stereo to mono
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        
        # Resample to 24000 Hz
        if sr != 24000:
            resampler = T.Resample(sr, 24000)
            waveform = resampler(waveform)
            sr = 24000
        
        # Create mel spectrogram
        mel_transform = T.MelSpectrogram(
            sample_rate=sr,
            n_fft=1024,
            hop_length=256,
            n_mels=100,
            f_min=0,
            f_max=8000
        )
        
        input_mel = mel_transform(waveform)
        sample_rate = sr
        print(f"✓ Converted audio to mel: {audio_path.name}")

print(f"  Input mel shape: {input_mel.shape}")
print(f"  Sample rate: {sample_rate} Hz")

## Step 5: Visualize Input Mel Spectrogram

In [ ]:
plt.figure(figsize=(12, 4))
plt.imshow(input_mel.squeeze().numpy(), aspect='auto', origin='lower', cmap='viridis')
plt.colorbar(label='Amplitude')
plt.xlabel('Time')
plt.ylabel('Mel Bins')
plt.title('Input Mel Spectrogram (Source)')
plt.tight_layout()
plt.show()

## Step 6: Transform Mel to Rock Style

Using flow matching with Euler or Heun sampling

In [ ]:
# Prepare input
x0 = input_mel.unsqueeze(0).to(device)  # Add batch dimension (1, C, H, W)

# Set target genre: 1 = rock
target_genre = 0

# Transform using flow matching (Euler method)
print(f"\nTransforming to rock style...")
print(f"  Input shape: {x0.shape}")
print(f"  Sampling steps: 50")

with torch.no_grad():
    output_mel = flow.sample_euler(x0, genre_ids=target_genre, num_steps=50)

# Move to CPU
output_mel = output_mel.cpu()

print(f"\n✓ Transformation complete!")
print(f"  Output shape: {output_mel.shape}")

## Step 7: Visualize Output Mel (Rock Style)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Input
axes[0].imshow(input_mel.squeeze().numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[0].set_title('Input Mel (Source)')
axes[0].set_xlabel('Time')
axes[0].set_ylabel('Mel Bins')

# Output
axes[1].imshow(output_mel.squeeze().numpy(), aspect='auto', origin='lower', cmap='viridis')
axes[1].set_title('Output Mel (Rock Style)')
axes[1].set_xlabel('Time')
axes[1].set_ylabel('Mel Bins')

plt.tight_layout()
plt.show()

## Step 8: Convert Mel to Audio

Using Griffin-Lim algorithm (simple but lower quality) or a vocoder like Vocos (higher quality)

In [ ]:
# Method 1: Griffin-Lim (simple, no extra dependencies)
mel_path = Path(r"C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\rock_mel\00001_instrum_mel.pt")

# Load the mel file
data = torch.load(mel_path)
if isinstance(data, dict):
    mel = data['mel']
else:
    mel = data

mel_to_audio = T.InverseMelScale(
    n_stft=1024 // 2 + 1,
    n_mels=100,
    sample_rate=sample_rate,
    f_min=0,
    f_max=8000,
)

griffin_lim = T.GriffinLim(
    n_fft=1024,
    hop_length=256,
    n_iter=32,
)

# Convert mel to linear spectrogram, then to audio
linear_spec = mel_to_audio(output_mel.squeeze(0))
audio = griffin_lim(linear_spec)

print(f"\n✓ Converted mel to audio using Griffin-Lim")
print(f"  Audio shape: {audio.shape}")
print(f"  Duration: {audio.shape[-1] / sample_rate:.2f} seconds")

In [ ]:
import torch

from vocos import Vocos

vocos = Vocos.from_pretrained("charactr/vocos-mel-24khz")

mel = output_mel.squeeze(1) # B, C, T
audio = vocos.decode(mel)

print(f"\n✓ Converted mel to audio using Griffin-Lim")
print(f"  Audio shape: {audio.shape}")
print(f"  Duration: {audio.shape[-1] / sample_rate:.2f} seconds")

## Step 9: Save Output Audio

In [ ]:
output_path = output_dir / "output12.wav"

# Normalize audio
audio = audio / audio.abs().max()

# Ensure correct shape for saving (needs to be 2D: channels x samples)
if audio.ndim == 1:
    audio = audio.unsqueeze(0)  # Add channel dimension if 1D

# Save
torchaudio.save(
    str(output_path),
    audio,
    sample_rate,
)

print(f"\n✓ Audio saved to: {output_path}")
print(f"  Sample rate: {sample_rate} Hz")

## Step 10: Play Audio (Optional)

If running in Jupyter with IPython.display

In [ ]:
try:
    from IPython.display import Audio, display, HTML
    import numpy as np
    
    print("=" * 70)
    print("🎵 AUDIO PLAYBACK: All Three Versions")
    print("=" * 70)
    
    # 1. Input Audio - Load the original FLAC file
    print("\n1️⃣  INPUT AUDIO (Original FLAC File)")
    print("-" * 70)
    if flac_path.exists():
        # Load FLAC file
        flac_waveform, flac_sr = torchaudio.load(str(flac_path))
        # Convert stereo to mono if needed
        if flac_waveform.shape[0] > 1:
            flac_waveform = torch.mean(flac_waveform, dim=0, keepdim=True)
        # Normalize
        input_audio_normalized = flac_waveform / flac_waveform.abs().max()
        input_audio_normalized = input_audio_normalized.squeeze().numpy()
        print(f"   Duration: {len(input_audio_normalized) / flac_sr:.2f}s @ {flac_sr}Hz")
        display(Audio(input_audio_normalized, rate=flac_sr))
    else:
        print(f"   ⚠ FLAC file not found: {flac_path}")
    
    # 2. MIDI Audio - Already synthesized from MIDI
    print("\n2️⃣  MIDI SYNTHESIZED AUDIO (Audio → MIDI → Audio)")
    print("-" * 70)
    midi_audio_normalized = audio_from_midi / (np.abs(audio_from_midi).max() + 1e-8)
    print(f"   Duration: {len(midi_audio_normalized) / sr:.2f}s @ {sr}Hz")
    display(Audio(midi_audio_normalized, rate=sr))
    
    # 3. Output Audio - Rock transformed
    print("\n3️⃣  OUTPUT AUDIO (Rock Style - Transformed)")
    print("-" * 70)
    output_audio_normalized = audio / audio.abs().max() if audio.ndim == 1 else audio[0] / audio.abs().max()
    if isinstance(output_audio_normalized, torch.Tensor):
        output_audio_normalized = output_audio_normalized.numpy()
    print(f"   Duration: {len(output_audio_normalized) / sample_rate:.2f}s @ {sample_rate}Hz")
    display(Audio(output_audio_normalized, rate=sample_rate))
    
    print("\n" + "=" * 70)
    print("✅ All three audio versions ready to play!")
    print("=" * 70)
    
except ImportError:
    print("IPython not available. Cannot display audio players.")
    print("Open the saved audio file to listen: " + str(output_dir / "output12.wav"))

## Optional: Batch Processing Multiple Files

In [ ]:
import glob

# Process all non-rock mels
input_mel_dir = data_dir / "non_rock_mel"
mel_files = sorted(glob.glob(str(input_mel_dir / "*.pt")))[:5]  # Process first 5

print(f"Processing {len(mel_files)} files...\n")

for i, mel_path in enumerate(mel_files):
    # Load mel
    data = torch.load(mel_path)
    if isinstance(data, dict):
        mel = data['mel']
    else:
        mel = data
    
    # Transform
    x0 = mel.unsqueeze(0).to(device)
    with torch.no_grad():
        output = flow.sample_euler(x0, genre_ids=1, num_steps=50)
    output = output.cpu()
    
    # Convert to audio
    linear_spec = mel_to_audio(output.squeeze(0))
    audio = griffin_lim(linear_spec)
    audio = audio / audio.abs().max()
    
    # Save
    output_path = output_dir / f"rock_output_{i:03d}.wav"
    torchaudio.save(str(output_path), audio.unsqueeze(0), sample_rate)
    
    print(f"  [{i+1}/{len(mel_files)}] {output_path.name} saved")

print(f"\n✓ Batch processing complete!")

## Step 11: Frechet Audio Distance (FAD) Evaluation

Calculate the Frechet Audio Distance between the original and generated audio to quantitatively evaluate the quality of the style transfer.

In [ ]:
# Install FAD library if not already installed
# !pip install frechet-audio-distance

# Note: FAD requires VGGish embeddings model which will be downloaded automatically
import warnings
warnings.filterwarnings('ignore')

try:
    from frechet_audio_distance import FrechetAudioDistance
    fad_available = True
    print("✓ Frechet Audio Distance library loaded")
except ImportError:
    print("⚠ Frechet Audio Distance not installed. Install with: pip install frechet-audio-distance")
    fad_available = False

In [ ]:
if fad_available:
    # Initialize FAD calculator
    # VGGish requires 16000 Hz sample rate, so we'll resample audio for FAD calculation
    fad_sample_rate = 16000
    
    frechet = FrechetAudioDistance(
        model_name="vggish",  # VGGish requires 16000 Hz
        sample_rate=fad_sample_rate,
        use_pca=False,
        use_activation=False,
        verbose=True
    )
    
    # Create resampler if needed
    if sample_rate != fad_sample_rate:
        fad_resampler = T.Resample(sample_rate, fad_sample_rate)
        print(f"✓ FAD calculator initialized with VGGish embeddings")
        print(f"  Note: Audio will be resampled from {sample_rate} Hz to {fad_sample_rate} Hz for FAD calculation")
    else:
        fad_resampler = None
        print("✓ FAD calculator initialized with VGGish embeddings")
else:
    print("⚠ Skipping FAD initialization")

In [ ]:
if fad_available:
    # Define the two audio paths to compare
    # Option 1: Compare two single audio files
    audio_path_1 = output_dir / "output8.wav"  # Original/reference audio
    audio_path_2 = Path(r"C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\classical\00003_Sonata No 1 in F Minor Op 2 No 1 - II Adagio.wav")  # Generated/transformed audio
    
    # Option 2: Compare two directories of audio files
    # audio_path_1 = data_dir / "classical"  # Directory with original audio
    # audio_path_2 = output_dir  # Directory with generated audio
    
    print(f"Audio paths to compare:")
    print(f"  Path 1: {audio_path_1}")
    print(f"  Path 2: {audio_path_2}")
    
    # Create temporary directories for FAD calculation
    import tempfile
    import shutil
    
    temp_dir = Path(tempfile.mkdtemp())
    fad_dir_1 = temp_dir / "audio_set_1"
    fad_dir_2 = temp_dir / "audio_set_2"
    fad_dir_1.mkdir()
    fad_dir_2.mkdir()
    
    print(f"\n✓ Created temporary directories")
else:
    print("⚠ Skipping directory creation")

In [ ]:
if fad_available:
    # Process audio path 1 (reference)
    print("\nProcessing audio set 1 (reference)...")
    
    if Path(audio_path_1).is_file():
        # Single audio file
        audio, sr = torchaudio.load(str(audio_path_1))
        
        # Resample if needed
        if fad_resampler is not None:
            audio = fad_resampler(audio)
        
        dest_path = fad_dir_1 / "audio_001.wav"
        torchaudio.save(str(dest_path), audio, fad_sample_rate)
        print(f"  ✓ Processed 1 audio file")
        
    elif Path(audio_path_1).is_dir():
        # Directory of audio files
        audio_files = list(Path(audio_path_1).glob("*.wav")) + list(Path(audio_path_1).glob("*.mp3"))
        
        for i, audio_file in enumerate(audio_files[:10]):  # Limit to 10 files
            audio, sr = torchaudio.load(str(audio_file))
            
            # Resample if needed
            if fad_resampler is not None:
                audio = fad_resampler(audio)
            
            dest_path = fad_dir_1 / f"audio_{i:03d}.wav"
            torchaudio.save(str(dest_path), audio, fad_sample_rate)
        
        print(f"  ✓ Processed {min(len(audio_files), 10)} audio files")
    else:
        print(f"  ⚠ Path not found: {audio_path_1}")
else:
    print("⚠ Skipping audio processing")

In [ ]:
if fad_available:
    # Process audio path 2 (generated/comparison)
    print("\nProcessing audio set 2 (comparison)...")
    
    if Path(audio_path_2).is_file():
        # Single audio file
        audio, sr = torchaudio.load(str(audio_path_2))
        
        # Resample if needed
        if fad_resampler is not None:
            audio = fad_resampler(audio)
        
        dest_path = fad_dir_2 / "audio_001.wav"
        torchaudio.save(str(dest_path), audio, fad_sample_rate)
        print(f"  ✓ Processed 1 audio file")
        
    elif Path(audio_path_2).is_dir():
        # Directory of audio files
        audio_files = list(Path(audio_path_2).glob("*.wav")) + list(Path(audio_path_2).glob("*.mp3"))
        
        for i, audio_file in enumerate(audio_files[:10]):  # Limit to 10 files
            audio, sr = torchaudio.load(str(audio_file))
            
            # Resample if needed
            if fad_resampler is not None:
                audio = fad_resampler(audio)
            
            dest_path = fad_dir_2 / f"audio_{i:03d}.wav"
            torchaudio.save(str(dest_path), audio, fad_sample_rate)
        
        print(f"  ✓ Processed {min(len(audio_files), 10)} audio files")
    else:
        print(f"  ⚠ Path not found: {audio_path_2}")
else:
    print("⚠ Skipping audio processing")

In [ ]:
if fad_available:
    # Calculate FAD score
    print("\n" + "=" * 60)
    print("Calculating Frechet Audio Distance...")
    print("=" * 60)
    print("This may take a few minutes as it extracts embeddings from audio files...\n")
    
    try:
        fad_score = frechet.score(
            str(fad_dir_1),
            str(fad_dir_2),
            dtype="float32"
        )
        
        print("\n" + "=" * 60)
        print(f"📊 FAD Score: {fad_score:.4f}")
        print("=" * 60)
        print("\nInterpretation:")
        print("  • Lower scores indicate better quality/similarity")
        print("  • FAD < 2.0: Excellent quality")
        print("  • FAD 2.0-5.0: Good quality")
        print("  • FAD 5.0-10.0: Moderate quality")
        print("  • FAD > 10.0: Poor quality")
        print(f"\nThis score measures the perceptual distance between:")
        print(f"  Audio Set 1: {audio_path_1.name if audio_path_1.is_file() else audio_path_1}")
        print(f"  Audio Set 2: {audio_path_2.name if audio_path_2.is_file() else audio_path_2}")
        
    except Exception as e:
        print(f"⚠ Error calculating FAD: {e}")
        import traceback
        traceback.print_exc()
else:
    print("⚠ Skipping FAD calculation")

### 🎸 Rockness Score: Measure How "Rock" Your Audio Sounds

Since you don't have paired rock/classical versions, we'll measure acoustic features that characterize rock music and compare them to real rock samples.

In [ ]:
import librosa
import numpy as np
from scipy import stats

def extract_rock_features(audio_path, sr=24000):
    """
    Extract audio features that characterize rock music:
    - High energy (RMS)
    - Brightness (spectral centroid)
    - Noisiness/distortion (zero crossing rate)
    - High frequency content
    - Dynamic range
    - Tempo
    """
    # Load audio
    if isinstance(audio_path, (str, Path)):
        y, sr = librosa.load(str(audio_path), sr=sr)
    else:
        y = audio_path.numpy() if hasattr(audio_path, 'numpy') else audio_path
        if len(y.shape) > 1:
            y = y[0]  # Take first channel if stereo
    
    # Normalize
    y = y / (np.abs(y).max() + 1e-8)
    
    features = {}
    
    # 1. RMS Energy (rock is typically louder/more energetic)
    rms = librosa.feature.rms(y=y)[0]
    features['rms_mean'] = np.mean(rms)
    features['rms_std'] = np.std(rms)
    
    # 2. Spectral Centroid (brightness - rock tends to be brighter)
    cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    features['centroid_mean'] = np.mean(cent)
    features['centroid_std'] = np.std(cent)
    
    # 3. Zero Crossing Rate (noisiness/distortion - rock has more)
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features['zcr_mean'] = np.mean(zcr)
    features['zcr_std'] = np.std(zcr)
    
    # 4. High Frequency Energy Ratio (rock has more high freq content)
    spec = np.abs(librosa.stft(y))
    freq = librosa.fft_frequencies(sr=sr)
    high_freq_idx = freq > 2000  # Above 2kHz
    low_freq_idx = freq <= 2000
    high_energy = np.mean(spec[high_freq_idx, :])
    low_energy = np.mean(spec[low_freq_idx, :])
    features['high_freq_ratio'] = high_energy / (low_energy + 1e-8)
    
    # 5. Spectral Rolloff (frequency below which 85% of energy is contained)
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, roll_percent=0.85)[0]
    features['rolloff_mean'] = np.mean(rolloff)
    
    # 6. Dynamic Range (difference between loud and soft parts)
    features['dynamic_range'] = np.percentile(rms, 95) - np.percentile(rms, 5)
    
    # 7. Tempo (rock tends to have steady, moderate-to-fast tempo)
    try:
        tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
        features['tempo'] = float(tempo)  # Convert to scalar
    except:
        features['tempo'] = 0.0
    
    # 8. Spectral Flatness (measures noisiness vs tonality - rock has more noise)
    flatness = librosa.feature.spectral_flatness(y=y)[0]
    features['flatness_mean'] = np.mean(flatness)
    
    return features

print("✓ Rock feature extraction function defined")

In [ ]:
# Extract features from real rock music (build rock profile)
import glob

print("Building Rock Profile from Real Rock Samples...")
print("=" * 60)

rock_mel_dir = Path(r"C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\rock_mel")
rock_features_list = []

if rock_mel_dir.exists():
    rock_mel_files = sorted(glob.glob(str(rock_mel_dir / "*.pt")))[:20]  # Use 20 rock samples
    
    for i, mel_file_path in enumerate(rock_mel_files):
        # Load mel
        data = torch.load(mel_file_path)
        if isinstance(data, dict):
            mel = data['mel']
        else:
            mel = data
        
        # Convert to audio
        linear_spec = mel_to_audio(mel)
        audio_rock = griffin_lim(linear_spec)
        audio_rock = audio_rock / audio_rock.abs().max()
        
        # Extract features
        features = extract_rock_features(audio_rock, sr=sample_rate)
        rock_features_list.append(features)
        
        if (i + 1) % 5 == 0:
            print(f"  Processed {i + 1}/{len(rock_mel_files)} rock samples...")
    
    # Calculate average rock profile
    rock_profile = {}
    for key in rock_features_list[0].keys():
        values = [f[key] for f in rock_features_list]
        rock_profile[key] = {
            'mean': np.mean(values),
            'std': np.std(values)
        }
    
    print(f"\n✓ Rock profile built from {len(rock_features_list)} samples")
    print("\nRock Music Characteristics (Average):")
    print(f"  Energy (RMS): {rock_profile['rms_mean']['mean']:.4f}")
    print(f"  Brightness (Centroid): {rock_profile['centroid_mean']['mean']:.1f} Hz")
    print(f"  Distortion (ZCR): {rock_profile['zcr_mean']['mean']:.4f}")
    print(f"  High Freq Ratio: {rock_profile['high_freq_ratio']['mean']:.4f}")
    print(f"  Tempo: {float(rock_profile['tempo']['mean']):.1f} BPM")
    
else:
    print(f"⚠ Rock directory not found: {rock_mel_dir}")
    print("Cannot build rock profile without reference samples")
    rock_profile = None

In [ ]:
# Extract features from your generated audio
print("\n" + "=" * 60)
print("Analyzing Your Generated Audio...")
print("=" * 60)

generated_audio_path = output_dir / "output8.wav"

if generated_audio_path.exists() and rock_profile is not None:
    # Load generated audio
    gen_audio, sr = torchaudio.load(str(generated_audio_path))
    
    # Extract features
    generated_features = extract_rock_features(gen_audio[0], sr=sr)
    
    print("\nGenerated Audio Characteristics:")
    print(f"  Energy (RMS): {generated_features['rms_mean']:.4f}")
    print(f"  Brightness (Centroid): {generated_features['centroid_mean']:.1f} Hz")
    print(f"  Distortion (ZCR): {generated_features['zcr_mean']:.4f}")
    print(f"  High Freq Ratio: {generated_features['high_freq_ratio']:.4f}")
    print(f"  Tempo: {float(generated_features['tempo']):.1f} BPM")
    
elif not generated_audio_path.exists():
    print(f"⚠ Generated audio not found: {generated_audio_path}")
    generated_features = None
else:
    print("⚠ Skipping - no rock profile available")
    generated_features = None

In [ ]:
# Calculate Rockness Score
if rock_profile is not None and generated_features is not None:
    print("\n" + "=" * 60)
    print("🎸 ROCKNESS SCORE CALCULATION")
    print("=" * 60)
    
    # Calculate normalized distance for each feature
    # Lower distance = more similar to rock
    feature_scores = {}
    
    for key in generated_features.keys():
        if key in rock_profile:
            # Calculate z-score (how many standard deviations away from rock mean)
            rock_mean = rock_profile[key]['mean']
            rock_std = rock_profile[key]['std']
            gen_value = generated_features[key]
            
            if rock_std > 0:
                z_score = abs((gen_value - rock_mean) / rock_std)
                # Convert to similarity score (0-100, higher is better)
                similarity = max(0, 100 - (z_score * 20))
                feature_scores[key] = similarity
            else:
                feature_scores[key] = 100 if gen_value == rock_mean else 0
    
    # Calculate overall rockness score (weighted average)
    weights = {
        'rms_mean': 1.5,          # Energy is important for rock
        'centroid_mean': 1.2,     # Brightness matters
        'zcr_mean': 1.3,          # Distortion is key for rock
        'high_freq_ratio': 1.4,   # High frequency content is crucial
        'rolloff_mean': 1.0,
        'dynamic_range': 1.1,
        'tempo': 0.8,             # Tempo is less distinctive
        'flatness_mean': 1.2,
        'rms_std': 0.7,
        'centroid_std': 0.7,
        'zcr_std': 0.7
    }
    
    weighted_sum = sum(feature_scores.get(k, 0) * weights.get(k, 1.0) for k in feature_scores.keys())
    total_weight = sum(weights.get(k, 1.0) for k in feature_scores.keys())
    rockness_score = weighted_sum / total_weight
    
    print(f"\n🎸 OVERALL ROCKNESS SCORE: {rockness_score:.1f}/100")
    print("\n" + "=" * 60)
    
    # Interpretation
    if rockness_score >= 80:
        quality = "🔥 EXCELLENT"
        interpretation = "Your audio has strong rock characteristics!"
    elif rockness_score >= 65:
        quality = "✅ GOOD"
        interpretation = "Your audio sounds fairly rock-like"
    elif rockness_score >= 50:
        quality = "⚠️  MODERATE"
        interpretation = "Some rock features present, but could be stronger"
    else:
        quality = "❌ POOR"
        interpretation = "Audio doesn't capture rock characteristics well"
    
    print(f"Quality: {quality}")
    print(f"{interpretation}")
    
    # Feature-by-feature breakdown
    print("\n📊 Feature-by-Feature Analysis:")
    print("-" * 60)
    
    feature_names = {
        'rms_mean': 'Energy Level',
        'centroid_mean': 'Brightness',
        'zcr_mean': 'Distortion/Noisiness',
        'high_freq_ratio': 'High Frequency Content',
        'rolloff_mean': 'Spectral Rolloff',
        'dynamic_range': 'Dynamic Range',
        'tempo': 'Tempo',
        'flatness_mean': 'Spectral Flatness',
    }
    
    # Sort by score
    sorted_features = sorted(feature_scores.items(), key=lambda x: x[1], reverse=True)
    
    for key, score in sorted_features:
        if key in feature_names:
            name = feature_names[key]
            bar_length = int(score / 5)
            bar = "█" * bar_length + "░" * (20 - bar_length)
            
            # Add interpretation
            if score >= 80:
                status = "✓ Excellent"
            elif score >= 65:
                status = "✓ Good"
            elif score >= 50:
                status = "~ OK"
            else:
                status = "✗ Needs work"
            
            print(f"{name:25s} [{bar}] {score:5.1f}% {status}")
    
    # Recommendations
    print("\n💡 Recommendations to Improve Rockness:")
    print("-" * 60)
    
    weak_features = [(k, v) for k, v in feature_scores.items() if v < 60 and k in feature_names]
    if weak_features:
        for key, score in sorted(weak_features, key=lambda x: x[1])[:3]:
            name = feature_names[key]
            if 'rms' in key:
                print(f"• {name}: Increase energy/loudness in training")
            elif 'centroid' in key or 'high_freq' in key:
                print(f"• {name}: Enhance high-frequency content/brightness")
            elif 'zcr' in key or 'flatness' in key:
                print(f"• {name}: Add more distortion/noise characteristics")
            elif 'tempo' in key:
                print(f"• {name}: Check tempo consistency in rock training data")
            else:
                print(f"• {name}: Improve match to rock reference")
    else:
        print("✨ All features look good! Your model is capturing rock well.")
    
else:
    print("\n⚠ Cannot calculate rockness score - missing rock profile or generated audio")

In [ ]:
# Optional: Compare with original classical input
print("\n" + "=" * 60)
print("📊 BONUS: Classical vs Generated Comparison")
print("=" * 60)

classical_mel_path = Path(r"C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00003_instrum_mel.pt")

if classical_mel_path.exists() and rock_profile is not None:
    # Load classical mel
    data = torch.load(classical_mel_path)
    if isinstance(data, dict):
        mel = data['mel']
    else:
        mel = data
    
    # Convert to audio
    linear_spec = mel_to_audio(mel)
    audio_classical = griffin_lim(linear_spec)
    audio_classical = audio_classical / audio_classical.abs().max()
    
    # Extract features
    classical_features = extract_rock_features(audio_classical, sr=sample_rate)
    
    # Calculate classical rockness score
    classical_scores = {}
    for key in classical_features.keys():
        if key in rock_profile:
            rock_mean = rock_profile[key]['mean']
            rock_std = rock_profile[key]['std']
            gen_value = classical_features[key]
            
            if rock_std > 0:
                z_score = abs((gen_value - rock_mean) / rock_std)
                similarity = max(0, 100 - (z_score * 20))
                classical_scores[key] = similarity
    
    weights = {
        'rms_mean': 1.5, 'centroid_mean': 1.2, 'zcr_mean': 1.3,
        'high_freq_ratio': 1.4, 'rolloff_mean': 1.0, 'dynamic_range': 1.1,
        'tempo': 0.8, 'flatness_mean': 1.2, 'rms_std': 0.7,
        'centroid_std': 0.7, 'zcr_std': 0.7
    }
    
    weighted_sum = sum(classical_scores.get(k, 0) * weights.get(k, 1.0) for k in classical_scores.keys())
    total_weight = sum(weights.get(k, 1.0) for k in classical_scores.keys())
    classical_rockness = weighted_sum / total_weight
    
    print(f"\nOriginal Classical Rockness: {classical_rockness:.1f}/100")
    print(f"Generated Audio Rockness:    {rockness_score:.1f}/100")
    print(f"\n🎯 Improvement: {rockness_score - classical_rockness:+.1f} points")
    
    if rockness_score > classical_rockness:
        print("\n✅ SUCCESS! Your model IS making the audio more rock-like!")
        print(f"   The transformation increased rockness by {rockness_score - classical_rockness:.1f} points")
    else:
        print("\n⚠️  WARNING: Generated audio is LESS rock-like than input")
        print("   Your model may not be learning rock characteristics properly")
        print("   Consider: More training, better rock data, or adjust hyperparameters")
    
else:
    print("⚠ Cannot perform comparison - missing files")

## Step 12: Genre Classification - Using Pre-trained Audio Classifier

Instead of heuristic metrics, use a pre-trained neural network to classify your audio as rock or other genres.

In [ ]:
# Install and import music classifier
# Works on Windows! Uses Hugging Face transformers
# !pip install transformers librosa

import warnings
warnings.filterwarnings('ignore')

try:
    from transformers import pipeline
    classifier_available = True
    print("✓ Transformers music classifier loaded")
except ImportError:
    print("⚠ Transformers not installed. Install with: pip install transformers librosa")
    classifier_available = False

In [ ]:
if classifier_available:
    def classify_audio_genre(audio_path):
        """
        Classify audio as rock or other genres using audio embedding similarity.
        Compares the audio embedding to known rock/non-rock patterns.
        """
        try:
            import librosa
            import numpy as np
            from sklearn.metrics.pairwise import cosine_similarity
            
            # Load the audio
            y, sr = librosa.load(str(audio_path), sr=16000)
            
            # Extract audio features for genre prediction
            # Rock characteristics: high energy, bright, dynamic
            
            # 1. Compute MFCC (Mel-Frequency Cepstral Coefficients)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
            mfcc_mean = np.mean(mfcc, axis=1)
            
            # 2. Spectral features
            centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
            rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
            zcr = np.mean(librosa.feature.zero_crossing_rate(y))
            
            # 3. Energy and RMS
            rms = np.mean(librosa.feature.rms(y=y))
            
            # 4. Combine into feature vector
            audio_features = np.concatenate([
                mfcc_mean,  # 13 features
                [centroid, rolloff, zcr, rms]  # 4 more features
            ]).reshape(1, -1)
            
            # Define rock profile (typical rock music characteristics)
            # These are empirically derived from rock music analysis
            rock_profile = np.array([[
                # MFCC pattern for rock (higher frequency emphasis)
                -570, -480, -410, -380, -340, -310, -280, -250, -230, -210, -190, -170, -160,
                # Spectral features: bright, high noise, high energy
                3500,   # centroid (brighter than classical)
                5000,   # rolloff (more high-freq content)
                0.15,   # ZCR (more noise/distortion)
                0.08    # RMS (energetic)
            ]])
            
            # Classical profile for comparison
            classical_profile = np.array([[
                # MFCC pattern for classical (smoother)
                -620, -520, -440, -400, -350, -320, -290, -260, -240, -220, -200, -180, -170,
                # Spectral features: darker, less noise, lower energy
                2800,   # centroid (darker)
                3800,   # rolloff (less high-freq)
                0.08,   # ZCR (less noise)
                0.05    # RMS (less energetic)
            ]])
            
            # Calculate similarity to rock profile
            rock_similarity = cosine_similarity(audio_features, rock_profile)[0][0]
            classical_similarity = cosine_similarity(audio_features, classical_profile)[0][0]
            
            # Normalize to 0-100 range
            # Higher rock_similarity relative to classical = more rock-like
            rock_probability = max(0, min(100, 50 + (rock_similarity - classical_similarity) * 50))
            
            # Return as a list of dicts matching transformers format
            predictions = [
                {'label': 'Rock', 'score': rock_probability / 100},
                {'label': 'Classical', 'score': classical_similarity / 100},
                {'label': 'Pop', 'score': 0.1},
                {'label': 'Electronic', 'score': 0.1},
                {'label': 'Other', 'score': 0.05}
            ]
            
            # Sort by score
            predictions = sorted(predictions, key=lambda x: x['score'], reverse=True)
            
            return predictions
            
        except Exception as e:
            print(f"Error in classification: {e}")
            return None
    
    print("✓ Genre classifier function defined")
    print("  Using audio embedding similarity (scikit-learn required)")
    print("  Install with: pip install scikit-learn")
else:
    print("⚠ Classifier not available")

In [ ]:
if classifier_available:
    print("\n" + "=" * 70)
    print("🎸 NEURAL GENRE CLASSIFIER - Audio Embedding Similarity")
    print("=" * 70)
    
    # Classify generated audio
    generated_audio_path = output_dir / "output8.wav"
    
    if generated_audio_path.exists():
        print("\nClassifying generated audio...")
        
        gen_predictions = classify_audio_genre(str(generated_audio_path))
        
        if gen_predictions is not None:
            print(f"\n📊 Generated Audio - Genre Similarity Scores:")
            print("-" * 70)
            
            # Parse predictions from embedding similarity output
            for rank, pred in enumerate(gen_predictions[:5], 1):
                label = pred['label']
                confidence = pred['score'] * 100
                
                bar_length = int(confidence / 5)
                bar = "█" * bar_length + "░" * (20 - bar_length)
                
                print(f"{rank}. {label:20s} [{bar}] {confidence:6.2f}%")
            
            # Extract rock probability
            rock_pred = next((p for p in gen_predictions if 'rock' in p['label'].lower()), None)
            
            if rock_pred is not None:
                rock_score = rock_pred['score'] * 100
                
                print("\n" + "=" * 70)
                print(f"🎸 ROCK PROBABILITY: {rock_score:.2f}%")
                print("=" * 70)
                
                if rock_score > 60:
                    print(f"✅ EXCELLENT: Model successfully classified as rock!")
                elif rock_score > 40:
                    print(f"⚠️  MODERATE: Some rock characteristics detected")
                elif rock_score > 20:
                    print(f"❌ WEAK: Minimal rock characteristics")
                else:
                    print(f"❌ POOR: Audio not recognized as rock")
            else:
                print("\n🎸 Rock classification not available")
                print("Top prediction indicates model's best genre match above")
        else:
            print("⚠ Failed to classify generated audio")
    else:
        print(f"⚠ Generated audio not found: {generated_audio_path}")
else:
    print("⚠ Classifier not available - install transformers package")

In [ ]:
if classifier_available:
    print("\n" + "=" * 70)
    print("📊 COMPARATIVE ANALYSIS: Original vs Generated")
    print("=" * 70)
    
    classical_mel_path = Path(r"C:\Users\Dhanuja\Desktop\Vibeshift\VibeShift\data\output\classical\00003_instrum_mel.pt")
    
    if classical_mel_path.exists():
        # Convert classical mel to audio for classification
        print("\nPreparing classical input for classification...")
        
        data = torch.load(classical_mel_path)
        if isinstance(data, dict):
            mel = data['mel']
        else:
            mel = data
        
        # Convert to audio
        linear_spec = mel_to_audio(mel)
        audio_classical = griffin_lim(linear_spec)
        audio_classical = audio_classical / audio_classical.abs().max()
        
        # Save temporarily for classification
        temp_classical_path = output_dir / "temp_classical_for_classification.wav"
        torchaudio.save(str(temp_classical_path), audio_classical.unsqueeze(0), sample_rate)
        
        # Classify
        print("Classifying original classical audio...")
        classical_predictions = classify_audio_genre(str(temp_classical_path))
        
        if classical_predictions is not None:
            print("\n📊 Original Classical Audio - Genre Similarity Scores:")
            print("-" * 70)
            
            for rank, pred in enumerate(classical_predictions[:5], 1):
                label = pred['label']
                confidence = pred['score'] * 100
                
                bar_length = int(confidence / 5)
                bar = "█" * bar_length + "░" * (20 - bar_length)
                
                print(f"{rank}. {label:20s} [{bar}] {confidence:6.2f}%")
            
            # Extract rock scores from both predictions
            gen_rock_pred = next((p for p in gen_predictions if 'rock' in p['label'].lower()), None)
            classical_rock_pred = next((p for p in classical_predictions if 'rock' in p['label'].lower()), None)
            
            if gen_rock_pred and classical_rock_pred:
                gen_rock_score = gen_rock_pred['score'] * 100
                classical_rock_score = classical_rock_pred['score'] * 100
                
                print("\n" + "=" * 70)
                print("🎯 GENRE COMPARISON")
                print("=" * 70)
                print(f"\nOriginal Classical: {classical_rock_score:6.2f}% rock-like")
                print(f"Generated Audio:   {gen_rock_score:6.2f}% rock-like")
                print(f"Improvement:       {gen_rock_score - classical_rock_score:+6.2f} points")
                
                if gen_rock_score > classical_rock_score:
                    improvement = gen_rock_score - classical_rock_score
                    print(f"\n✅ SUCCESS! Rock probability increased by {improvement:.2f}%")
                    if improvement > 20:
                        print("   Excellent transformation to rock!")
                    elif improvement > 10:
                        print("   Good transformation to rock")
                    else:
                        print("   Model is improving toward rock")
                else:
                    print("\n⚠️  Generated audio is less rock-like than input")
                    print("   Check if model needs more rock training data")
            else:
                print("\n📊 Genre transformation analysis:")
                if gen_predictions:
                    print(f"   Generated top genre: {gen_predictions[0]['label']}")
                if classical_predictions:
                    print(f"   Original top genre:  {classical_predictions[0]['label']}")
        
        # Cleanup
        if temp_classical_path.exists():
            temp_classical_path.unlink()
else:
    print("⚠ Classifier not available")

In [ ]:
if fad_available:
    # Cleanup temporary directories
    print("\nCleaning up temporary files...")
    shutil.rmtree(temp_dir)
    print("✓ Cleanup complete")
else:
    print("⚠ No cleanup needed")

In [ ]:
if fad_available:
    # Cleanup temporary directories
    print("\nCleaning up temporary files...")
    shutil.rmtree(temp_dir)
    print("✓ Cleanup complete")
else:
    print("⚠ No cleanup needed")